# Privacy, PII and Data Governance Analysis
## NovaCred Credit Application Dataset - Governance Officer 

#### We will, in the following notebook, deeply analyse the potential risks for privacy and gorvernance of the NovaCred credit application dataset. By setting ourselves different objectives such as pin pointing personal identifiable informations, map findings to GDPR requirements and more, we will aim to propose governance improvements for the NovaCard system. 

In [2]:
import pandas as pd
import json
from pathlib import Path

# Define file path
data_path = Path("../Data/raw_credit_applications.json")

# Load raw JSON file
with open(data_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(raw_data)

# Basic preview
print(f"Number of records: {len(df)}")
print(f"Columns: {list(df.columns)}")

df.head()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Number of records: 502
Columns: ['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp', 'loan_purpose', 'notes']


,_id,applicant_info,financials,spending_behavior,decision,processing_timestamp,loan_purpose,notes
0,app_200,"{'full_name': 'Jerry Smith', 'email': 'jerry.s...","{'annual_income': 73000, 'credit_history_month...","[{'category': 'Shopping', 'amount': 480}, {'ca...","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN
1,app_037,"{'full_name': 'Brandon Walker', 'email': 'bran...","{'annual_income': 78000, 'credit_history_month...","[{'category': 'Rent', 'amount': 608}, {'catego...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
2,app_215,"{'full_name': 'Scott Moore', 'email': 'scott.m...","{'annual_income': 61000, 'credit_history_month...","[{'category': 'Rent', 'amount': 109}]","{'loan_approved': True, 'interest_rate': 3.7, ...",NaN,vacation,NaN
3,app_024,"{'full_name': 'Thomas Lee', 'email': 'thomas.l...","{'annual_income': 103000, 'credit_history_mont...","[{'category': 'Fitness', 'amount': 575}]","{'loan_approved': True, 'interest_rate': 4.3, ...",NaN,NaN,NaN
4,app_184,"{'full_name': 'Brian Rodriguez', 'email': 'bri...","{'annual_income': 57000, 'credit_history_month...","[{'category': 'Entertainment', 'amount': 463}]","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN


#### We flatten the data set, as it contains a nested structure, to use it in a more effective way accross the rest of our analyses. 

In [3]:
applicant_df = pd.json_normalize(df["applicant_info"]).add_prefix("applicant_")
financials_df = pd.json_normalize(df["financials"]).add_prefix("financial_")
decision_df = pd.json_normalize(df["decision"]).add_prefix("decision_")

df_flat = pd.concat(
    [
        df["_id"],
        applicant_df,
        financials_df,
        decision_df,
        df["spending_behavior"],
        df["processing_timestamp"],
        df["loan_purpose"],
        df["notes"],
    ],
    axis=1,
)

df_flat.head()

,_id,applicant_full_name,applicant_email,applicant_ssn,applicant_ip_address,applicant_gender,applicant_date_of_birth,applicant_zip_code,financial_annual_income,financial_credit_history_months,...,financial_savings_balance,financial_annual_salary,decision_loan_approved,decision_rejection_reason,decision_interest_rate,decision_approved_amount,spending_behavior,processing_timestamp,loan_purpose,notes
0,app_200,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,...,31212,NaN,False,algorithm_risk_score,NaN,NaN,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,NaN,NaN
1,app_037,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,...,17915,NaN,False,algorithm_risk_score,NaN,NaN,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,NaN,NaN
2,app_215,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,...,37909,NaN,True,NaN,3.7,59000.0,"[{'category': 'Rent', 'amount': 109}]",NaN,vacation,NaN
3,app_024,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000,70,...,0,NaN,True,NaN,4.3,34000.0,"[{'category': 'Fitness', 'amount': 575}]",NaN,NaN,NaN
4,app_184,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,57000,14,...,31763,NaN,False,algorithm_risk_score,NaN,NaN,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,NaN,NaN


### PII Identification 